In [67]:
import pandas as pd
import torch

# split 80 20
from sklearn.model_selection import train_test_split

# create dataloader
from torch.utils.data import DataLoader

In [ ]:
# load data from ./data/titanic/train.csv and ./data/titanic/test.csv


train_df = pd.read_csv("./data/titanic/train.csv")

# Fill missing values in 'Embarked' with the mode
train_df["Embarked"] = train_df["Embarked"].fillna(train_df["Embarked"].mode()[0])

# convert Parch to categorical with dummies
train_df = pd.get_dummies(train_df, columns=["Parch"])

# convert SibSp to categorical with dummies
train_df = pd.get_dummies(train_df, columns=["SibSp"])

# convert Embarked to categorical with dummies
train_df = pd.get_dummies(train_df, columns=["Embarked"])

# convert Pclass to categorical with dummies
train_df = pd.get_dummies(train_df, columns=["Pclass"])


def preprocess_data(df):
    # drop passenger id, name, ticket, cabin
    df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1, inplace=True)
    # Fill missing values in 'Age' with the median age
    df["Age"] = df["Age"].fillna(df["Age"].median())

    # change survived to 0/1
    if "Survived" in df.columns:
        df["Survived"] = df["Survived"].map({0: 0, 1: 1})

    # Fill missing values in 'Fare' with the median fare
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    # convert Sex to 0/1
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

    # cast boolean dummy columns to int (0/1) for PyTorch compatibility
    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(int)

    # standardize Age and Fare
    df["Age"] = (df["Age"] - df["Age"].mean()) / df["Age"].std()
    df["Fare"] = (df["Fare"] - df["Fare"].mean()) / df["Fare"].std()

    return df


train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

# preprocess data
train_df = preprocess_data(train_df)
val_df = preprocess_data(val_df)
# assert that train and val have same columns
assert set(train_df.columns) == set(val_df.columns), (
    "Train and val have different columns"
    + str(set(train_df.columns) - set(val_df.columns))
)

X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]


# design model
model = torch.nn.Sequential(
    torch.nn.Linear(X.shape[1], 10),
    torch.nn.ReLU(),
    torch.nn.Linear(10, 10),
    torch.nn.ReLU(),
    torch.nn.Linear(10, 1),
    torch.nn.Sigmoid(),
)


train_ds = torch.utils.data.TensorDataset(
    torch.tensor(X.values, dtype=torch.float32),
    torch.tensor(y.values, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# train model
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = torch.nn.BCELoss()

n_epochs = 10000
for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch.unsqueeze(1))
        loss.backward()
        optimizer.step()


# print number of parameters in model
n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in model: {n_params}")

# evaluate model on validation set
X_val = val_df.drop("Survived", axis=1)
y_val = val_df["Survived"]
with torch.no_grad():
    y_val_pred = model(torch.tensor(X_val.values, dtype=torch.float32))
    y_val_pred = (y_val_pred > 0.5).float()
    accuracy = (y_val_pred.squeeze() == torch.tensor(y_val.values)).float().mean()
    print(f"Validation accuracy: {accuracy:.4f}")

Number of parameters in model: 361
Validation accuracy: 0.7877
